In [ ]:
import pandas as pd
import numpy as np
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score, confusion_matrix
import time
import re
import warnings
warnings.filterwarnings('ignore')

performance = pd.DataFrame(columns=['Name', 'Accuracy', 'Precision', 'Sensitivity', 'F1 Score', 'MCC', 'Markedness', "Youden's J", 'FMI', 'Time'])

originalDF = pd.read_csv("StealthPhisher_V3.csv")

LABEL = originalDF.iloc[:,-1:].columns[0]

fs = ['LengthOfURL', 'URLComplexity', 'CharacterComplexity',
       'DomainLengthOfURL', 'IsDomainIP', 'TLDLength', 'LetterCntInURL',
       'URLLetterRatio', 'DigitCntInURL', 'URLDigitRatio', 'EqualCharCntInURL',
       'QuesMarkCntInURL', 'AmpCharCntInURL', 'OtherSpclCharCntInURL',
       'URLOtherSpclCharRatio', 'NumberOfHashtags', 'NumberOfSubdomains',
       'HavingPath', 'Path Length', 'HavingQuery', 'HavingFragment',
       'HavingAnchor', 'HasSSL', 'IsUnreachable', 'LineOfCode',
       'LongestLineLength', 'HasTitle', 'TitleMatchScore', 'HasFavicon',LABEL]
df = pd.DataFrame(originalDF[fs]).copy()
print('Dataset Shape: ', df.shape)
df.iloc[:,-1:].value_counts()

In [ ]:
MCH = pd.DataFrame(columns=['URL', 'y_pred1','y_pred2','y_pred3','y_pred4','y_pred5','Label'])
MCH['URL'] = originalDF['URL']

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import tensorflow.keras.backend as K

X_train = df.drop(columns=[LABEL])
X_test = df.drop(columns=[LABEL])
y_train = df[LABEL]
y_test = df[LABEL]

# Define custom metrics
def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)

# Build Feed-Forward Neural Network (FNN)
def create_fnn(input_dim):
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build Attention-based MLP
class AttentionMLP(tf.keras.Model):
    def __init__(self, input_dim):
        super(AttentionMLP, self).__init__()
        self.input_layer = layers.InputLayer(input_shape=(input_dim,))
        self.cast_to_float = layers.Lambda(lambda x: tf.cast(x, tf.float32))  # Ensure float32
        self.attention_weights = layers.Dense(input_dim, activation='softmax')
        self.hidden_layer = layers.Dense(128, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.output_layer = layers.Dense(1, activation='sigmoid')

    def call(self, inputs):
        inputs = self.cast_to_float(inputs)  # Cast inputs to float32
        weights = self.attention_weights(inputs)
        weighted_inputs = inputs * weights
        x = self.hidden_layer(weighted_inputs)
        x = self.dropout(x)
        return self.output_layer(x)

# Train FNN
fnn = create_fnn(X_train.shape[1])
fnn.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Train Attention-based MLP
attention_mlp = AttentionMLP(X_train.shape[1])
attention_mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
attention_mlp.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Stacking: Combine results from FNN and Attention-based MLP
class StackingMetaClassifier:
    def __init__(self, fnn_model, attention_mlp_model):
        self.fnn_model = fnn_model
        self.attention_mlp_model = attention_mlp_model
        self.meta_model = LogisticRegression()

    def fit(self, X, y):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        self.meta_model.fit(meta_features, y)

    def predict(self, X):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        return self.meta_model.predict(meta_features)

stacked_model = StackingMetaClassifier(fnn, attention_mlp)
stacked_model.fit(X_train, y_train)
MCH['y_pred1'] = stacked_model.predict(X_test)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import tensorflow.keras.backend as K

X_train = df.drop(columns=[LABEL])
X_test = df.drop(columns=[LABEL])
y_train = df[LABEL]
y_test = df[LABEL]

# Define custom metrics
def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)

# Build Feed-Forward Neural Network (FNN)
def create_fnn(input_dim):
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build Attention-based MLP
class AttentionMLP(tf.keras.Model):
    def __init__(self, input_dim):
        super(AttentionMLP, self).__init__()
        self.input_layer = layers.InputLayer(input_shape=(input_dim,))
        self.cast_to_float = layers.Lambda(lambda x: tf.cast(x, tf.float32))  # Ensure float32
        self.attention_weights = layers.Dense(input_dim, activation='softmax')
        self.hidden_layer = layers.Dense(128, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.output_layer = layers.Dense(1, activation='sigmoid')

    def call(self, inputs):
        inputs = self.cast_to_float(inputs)  # Cast inputs to float32
        weights = self.attention_weights(inputs)
        weighted_inputs = inputs * weights
        x = self.hidden_layer(weighted_inputs)
        x = self.dropout(x)
        return self.output_layer(x)

# Train FNN
fnn = create_fnn(X_train.shape[1])
fnn.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Train Attention-based MLP
attention_mlp = AttentionMLP(X_train.shape[1])
attention_mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
attention_mlp.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Stacking: Combine results from FNN and Attention-based MLP
class StackingMetaClassifier:
    def __init__(self, fnn_model, attention_mlp_model):
        self.fnn_model = fnn_model
        self.attention_mlp_model = attention_mlp_model
        self.meta_model = LogisticRegression()

    def fit(self, X, y):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        self.meta_model.fit(meta_features, y)

    def predict(self, X):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        return self.meta_model.predict(meta_features)

stacked_model = StackingMetaClassifier(fnn, attention_mlp)
stacked_model.fit(X_train, y_train)
MCH['y_pred2'] = stacked_model.predict(X_test)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import tensorflow.keras.backend as K

X_train = df.drop(columns=[LABEL])
X_test = df.drop(columns=[LABEL])
y_train = df[LABEL]
y_test = df[LABEL]

# Define custom metrics
def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)

# Build Feed-Forward Neural Network (FNN)
def create_fnn(input_dim):
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build Attention-based MLP
class AttentionMLP(tf.keras.Model):
    def __init__(self, input_dim):
        super(AttentionMLP, self).__init__()
        self.input_layer = layers.InputLayer(input_shape=(input_dim,))
        self.cast_to_float = layers.Lambda(lambda x: tf.cast(x, tf.float32))  # Ensure float32
        self.attention_weights = layers.Dense(input_dim, activation='softmax')
        self.hidden_layer = layers.Dense(128, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.output_layer = layers.Dense(1, activation='sigmoid')

    def call(self, inputs):
        inputs = self.cast_to_float(inputs)  # Cast inputs to float32
        weights = self.attention_weights(inputs)
        weighted_inputs = inputs * weights
        x = self.hidden_layer(weighted_inputs)
        x = self.dropout(x)
        return self.output_layer(x)

# Train FNN
fnn = create_fnn(X_train.shape[1])
fnn.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Train Attention-based MLP
attention_mlp = AttentionMLP(X_train.shape[1])
attention_mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
attention_mlp.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Stacking: Combine results from FNN and Attention-based MLP
class StackingMetaClassifier:
    def __init__(self, fnn_model, attention_mlp_model):
        self.fnn_model = fnn_model
        self.attention_mlp_model = attention_mlp_model
        self.meta_model = LogisticRegression()

    def fit(self, X, y):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        self.meta_model.fit(meta_features, y)

    def predict(self, X):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        return self.meta_model.predict(meta_features)

stacked_model = StackingMetaClassifier(fnn, attention_mlp)
stacked_model.fit(X_train, y_train)

stacked_model = StackingMetaClassifier(fnn, attention_mlp)
stacked_model.fit(X_train, y_train)
MCH['y_pred3'] = stacked_model.predict(X_test)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import tensorflow.keras.backend as K

X_train = df.drop(columns=[LABEL])
X_test = df.drop(columns=[LABEL])
y_train = df[LABEL]
y_test = df[LABEL]

# Define custom metrics
def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)

# Build Feed-Forward Neural Network (FNN)
def create_fnn(input_dim):
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build Attention-based MLP
class AttentionMLP(tf.keras.Model):
    def __init__(self, input_dim):
        super(AttentionMLP, self).__init__()
        self.input_layer = layers.InputLayer(input_shape=(input_dim,))
        self.cast_to_float = layers.Lambda(lambda x: tf.cast(x, tf.float32))  # Ensure float32
        self.attention_weights = layers.Dense(input_dim, activation='softmax')
        self.hidden_layer = layers.Dense(128, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.output_layer = layers.Dense(1, activation='sigmoid')

    def call(self, inputs):
        inputs = self.cast_to_float(inputs)  # Cast inputs to float32
        weights = self.attention_weights(inputs)
        weighted_inputs = inputs * weights
        x = self.hidden_layer(weighted_inputs)
        x = self.dropout(x)
        return self.output_layer(x)

# Train FNN
fnn = create_fnn(X_train.shape[1])
fnn.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Train Attention-based MLP
attention_mlp = AttentionMLP(X_train.shape[1])
attention_mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
attention_mlp.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Stacking: Combine results from FNN and Attention-based MLP
class StackingMetaClassifier:
    def __init__(self, fnn_model, attention_mlp_model):
        self.fnn_model = fnn_model
        self.attention_mlp_model = attention_mlp_model
        self.meta_model = LogisticRegression()

    def fit(self, X, y):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        self.meta_model.fit(meta_features, y)

    def predict(self, X):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        return self.meta_model.predict(meta_features)

stacked_model = StackingMetaClassifier(fnn, attention_mlp)
stacked_model.fit(X_train, y_train)
MCH['y_pred4'] = stacked_model.predict(X_test)

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef, roc_auc_score
)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
import tensorflow.keras.backend as K

X_train = df.drop(columns=[LABEL])
X_test = df.drop(columns=[LABEL])
y_train = df[LABEL]
y_test = df[LABEL]

# Define custom metrics
def markedness(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return precision + specificity - 1

def youdens_j(y_true, y_pred):
    sensitivity = recall_score(y_true, y_pred)
    specificity = recall_score(y_true, y_pred, pos_label=0)
    return sensitivity + specificity - 1

def fowlkes_mallows_index(y_true, y_pred):
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    return np.sqrt(precision * recall)

# Build Feed-Forward Neural Network (FNN)
def create_fnn(input_dim):
    model = models.Sequential([
        layers.Dense(128, activation='relu', input_shape=(input_dim,)),
        layers.Dropout(0.3),
        layers.Dense(64, activation='relu'),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
    return model

# Build Attention-based MLP
class AttentionMLP(tf.keras.Model):
    def __init__(self, input_dim):
        super(AttentionMLP, self).__init__()
        self.input_layer = layers.InputLayer(input_shape=(input_dim,))
        self.cast_to_float = layers.Lambda(lambda x: tf.cast(x, tf.float32))  # Ensure float32
        self.attention_weights = layers.Dense(input_dim, activation='softmax')
        self.hidden_layer = layers.Dense(128, activation='relu')
        self.dropout = layers.Dropout(0.3)
        self.output_layer = layers.Dense(1, activation='sigmoid')

    def call(self, inputs):
        inputs = self.cast_to_float(inputs)  # Cast inputs to float32
        weights = self.attention_weights(inputs)
        weighted_inputs = inputs * weights
        x = self.hidden_layer(weighted_inputs)
        x = self.dropout(x)
        return self.output_layer(x)

# Train FNN
fnn = create_fnn(X_train.shape[1])
fnn.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Train Attention-based MLP
attention_mlp = AttentionMLP(X_train.shape[1])
attention_mlp.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
attention_mlp.fit(X_train, y_train, epochs=10, batch_size=32, validation_split=0.2, verbose=1)

# Stacking: Combine results from FNN and Attention-based MLP
class StackingMetaClassifier:
    def __init__(self, fnn_model, attention_mlp_model):
        self.fnn_model = fnn_model
        self.attention_mlp_model = attention_mlp_model
        self.meta_model = LogisticRegression()

    def fit(self, X, y):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        self.meta_model.fit(meta_features, y)

    def predict(self, X):
        fnn_preds = self.fnn_model.predict(X).ravel()
        mlp_preds = self.attention_mlp_model.predict(X).ravel()
        meta_features = np.stack([fnn_preds, mlp_preds], axis=1)
        return self.meta_model.predict(meta_features)

stacked_model = StackingMetaClassifier(fnn, attention_mlp)
stacked_model.fit(X_train, y_train)
MCH['y_pred5'] = stacked_model.predict(X_test)

In [ ]:
MCH['Label'] = originalDF['Label']
MCH.head(5)
MCH.to_csv('Final_MCH.csv')